# Download the source datasets

This notebook downloads the public source datasets into the root-level `data/raw/` folder. It works when launched from the project root or from `notebooks/`.

In [1]:
%pip install -q kagglehub

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from zipfile import ZipFile
import kagglehub
import requests

datasets = {
    'induction_motor': 'uysalserkan/fault-induction-motor-dataset',
    'gearbox': 'brjapon/gearbox-fault-diagnosis',
}
mendeley_datasets = {
    'jaka_minicobo_friction': 'https://data.mendeley.com/public-api/zip/n3t2bgsfcv/download/1',
}

project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data_dir = project_dir / 'data' / 'raw'
data_dir.mkdir(parents=True, exist_ok=True)

for name, dataset in datasets.items():
    destination = data_dir / name
    if destination.exists() and any(destination.iterdir()):
        print(f'Skipped {name}: files already exist in {destination}')
        continue
    path = kagglehub.dataset_download(dataset, output_dir=str(destination))
    print(f'Downloaded {name}: {path}')

for name, url in mendeley_datasets.items():
    destination = data_dir / name
    files = [file for file in destination.rglob('*') if file.is_file()] if destination.exists() else []
    if files:
        print(f'Skipped {name}: files already exist in {destination}')
        continue
    archive = data_dir / f'{name}.zip'
    if not archive.exists():
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with archive.open('wb') as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
    destination.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive) as zip_file:
        zip_file.extractall(destination)
    print(f'Downloaded {name}: {destination}')

Skipped induction_motor: files already exist in D:\P-SYNTH Physics Guided Synthetic Maintenance Data Generator\data\raw\induction_motor
Skipped gearbox: files already exist in D:\P-SYNTH Physics Guided Synthetic Maintenance Data Generator\data\raw\gearbox


Downloaded jaka_minicobo_friction: D:\P-SYNTH Physics Guided Synthetic Maintenance Data Generator\data\raw\jaka_minicobo_friction


In [3]:
for name in [*datasets, *mendeley_datasets]:
    folder = data_dir / name
    files = [file for file in folder.rglob('*') if file.is_file()]
    assert files, f'No files found for {name}'
    print(f'{name}: {len(files)} file(s) in {folder}')

induction_motor: 383 file(s) in D:\P-SYNTH Physics Guided Synthetic Maintenance Data Generator\data\raw\induction_motor
gearbox: 21 file(s) in D:\P-SYNTH Physics Guided Synthetic Maintenance Data Generator\data\raw\gearbox
jaka_minicobo_friction: 2 file(s) in D:\P-SYNTH Physics Guided Synthetic Maintenance Data Generator\data\raw\jaka_minicobo_friction
